# Universal Object Tracking Pipeline
**Give acces to your camera first because it will force you to reload website**


This notebook runs a real-time object detection and tracking pipeline inside Google Colab using your webcam.

It uses the nvidia/LocateAnything-3B vision-language model to locate items directly on your hardware.

## How to Run
1) Verify GPU Hardware: Ensure your Colab runtime is set to a T4 GPU (Runtime > Change runtime type > T4 GPU).

2) Execute Cell 1 & 2: Install dependencies and load the model weights into memory.

3) Run Cell 3: Start the embedded web application widget.

## Track Targets:
Type what you want to find into the text input box (e.g., cup, a black cup, foam), then click Start Tracking. Leaving the box blank defaults to tracking human faces and eyes.

## Workflow Layout
Cell 1 (Setup): Installs required packages. Run once per session.

Cell 2 (Model initialization): Loads the 3B parameter model into your GPU VRAM. Run once.

Cell 3 (Interactive Interface): Contains the Python-to-JavaScript bridge and the camera interface. Run this cell whenever you want to reset or update the web application tracking view.


In [ ]:
# ═══════════════════════════════════════════════════════
# 1. INSTALL
# ═══════════════════════════════════════════════════════
!pip install -q transformers==4.57.1 decord==0.6.0 lmdb==1.7.5
print('✅ Done.')

In [ ]:
# ═══════════════════════════════════════════════════════
# 2. WORKER ARCHITECTURE
# ═══════════════════════════════════════════════════════
import re, io, time, base64
import torch
from PIL import Image
from transformers import AutoModel, AutoTokenizer, AutoProcessor
from google.colab import userdata
from google.colab import output


class LocateAnythingWorker:
    def __init__(self, model_path: str, device: str = "cuda", dtype=torch.bfloat16):
        self.device = device
        self.dtype = dtype
        self.tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
        self.processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
        self.model = AutoModel.from_pretrained(
            model_path, torch_dtype=dtype, trust_remote_code=True,
            low_cpu_mem_usage=True, device_map="cuda" if device == "cuda" else "cpu"
        ).eval()

    @torch.no_grad()
    def predict(self, image: Image.Image, question: str) -> str:
        messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": question}]}]
        text = self.processor.py_apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        images, videos = self.processor.process_vision_info(messages)
        inputs = self.processor(text=[text], images=images, videos=videos, return_tensors="pt").to(self.device)

        response = self.model.generate(
            pixel_values=inputs["pixel_values"].to(self.dtype), input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"], image_grid_hws=inputs.get("image_grid_hws", None),
            tokenizer=self.tokenizer, max_new_tokens=1024, use_cache=True,
            generation_mode="hybrid", temperature=0.7, do_sample=True, top_p=0.9, repetition_penalty=1.1, verbose=False
        )
        return response[0] if isinstance(response, tuple) else response

    @staticmethod
    def parse_combined_boxes(answer, image_width, image_height):
        boxes = []
        current_label = None
        for m in re.finditer(r"<ref>(.*?)</ref>|<box><(\d+)><(\d+)><(\d+)><(\d+)></box>", answer):
            label, x1, y1, x2, y2 = m.groups()
            if label is not None:
                current_label = label.strip().lower()
                continue
            if current_label is not None:
                boxes.append({
                    "label": current_label,
                    "x1": int(x1) / 1000 * image_width, "y1": int(y1) / 1000 * image_height,
                    "x2": int(x2) / 1000 * image_width, "y2": int(y2) / 1000 * image_height,
                })
        return boxes

# ═══════════════════════════════════════════════════════
# 4. LOAD MODEL INTO GPU
# ═══════════════════════════════════════════════════════
HF_TOKEN = userdata.get('HF_TOKEN')
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)

MODEL_ID = 'nvidia/LocateAnything-3B'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.bfloat16 if DEVICE == 'cuda' else torch.float32

print(f'🔧 Loading {MODEL_ID} on {DEVICE}...')
worker = LocateAnythingWorker(MODEL_ID, device=DEVICE, dtype=DTYPE)
print('✅ Model loaded and ready.')



In [ ]:
# ═══════════════════════════════════════════════════════
# 5. Live Application Bridge & UI Layout
# ═══════════════════════════════════════════════════════
def process_frame_bridge(base64_image_string, targets_string):
    """Invoked directly by the cell's JavaScript runtime engine."""
    try:
        t_list = [x.strip() for x in targets_string.split(',') if x.strip()]
        if not t_list:
            t_list = ["closed eye", "open eye", "human face"]

        img = Image.open(io.BytesIO(base64.b64decode(base64_image_string))).convert('RGB')
        if max(img.size) > 640:
            s = 640 / max(img.size)
            img = img.resize((int(img.width*s), int(img.height*s)), Image.LANCZOS)

        joined_targets = " </c> ".join(t_list)
        prompt = f"Locate all the instances that match the following descriptions: {joined_targets}."

        start_t = time.time()
        model_output = worker.predict(img, prompt)
        detections = worker.parse_combined_boxes(model_output, img.width, img.height)

        print(f"⏱️ Model processing time: {time.time() - start_t:.2f}s")
        return output.eval_js(f"window.renderDetections({detections})")
    except Exception as e:
        return output.eval_js(f"console.error('Python Error: {str(e)}')")

# Register the Python function so JS can see it
output.register_callback('notebook.process_frame_bridge', process_frame_bridge)

# ═══════════════════════════════════════════════════════
#  INJECT EMBEDDED WEB INTERFACE
# ═══════════════════════════════════════════════════════
from IPython.display import HTML, display

INTERFACE_HTML = r'''
<div style="font-family:'Segoe UI',sans-serif; text-align:center; background:#121212; color:#e0e0e0; padding:15px; border-radius:10px;">
    <h3>🎯 Embedded Low-Latency Tracking Widget</h3>
    <div style="margin-bottom: 15px;">
        <input type="text" id="target_input" value="cup, a black cup, foam" style="background:#2d2d2d; color:#fff; border:1px solid #444; padding:8px; width:340px; border-radius:4px;">
        <button id="toggle_btn" onclick="togglePipeline()" style="background:#007acc; color:#fff; border:none; padding:8px 16px; border-radius:4px; cursor:pointer; font-weight:bold;">Start Tracking</button>
    </div>
    <div id="status_indicator" style="font-size:20px; font-weight:bold; margin-bottom:10px; color:#888;">Status: Stopped</div>
    <div style="position:relative; display:inline-block; background:#000; border-radius:8px; overflow:hidden;">
        <video id="webcam_stream" width="640" height="480" autoplay playsinline style="transform:scaleX(-1); display:block;"></video>
        <canvas id="overlay_canvas" width="640" height="480" style="position:absolute; top:0; left:0; transform:scaleX(-1);"></canvas>
    </div>
</div>

<script>
let active = false;
const video = document.getElementById('webcam_stream');
const canvas = document.getElementById('overlay_canvas');
const ctx = canvas.getContext('2d');
const status = document.getElementById('status_indicator');
const btn = document.getElementById('toggle_btn');

navigator.mediaDevices.getUserMedia({video:{width:640, height:480}})
    .then(stream => { video.srcObject = stream; })
    .catch(err => { alert('Webcam access denied: ' + err); });

function togglePipeline() {
    active = !active;
    if(active) {
        btn.innerText = "Stop Tracking";
        btn.style.background = "#f33";
        status.innerText = "Tracking Active";
        status.style.color = "#00bcff";
        captureCycle();
    } else {
        btn.innerText = "Start Tracking";
        btn.style.background = "#007acc";
        status.innerText = "Status: Stopped";
        status.style.color = "#888";
        ctx.clearRect(0, 0, canvas.width, canvas.height);
    }
}

async function captureCycle() {
    if (!active) return;

    // Draw frame onto temporary scaling canvas
    let hiddenCanvas = document.createElement('canvas');
    hiddenCanvas.width = video.videoWidth || 640;
    hiddenCanvas.height = video.videoHeight || 480;
    hiddenCanvas.getContext('2d').drawImage(video, 0, 0, hiddenCanvas.width, hiddenCanvas.height);

    let base64Image = hiddenCanvas.toDataURL('image/jpeg', 0.85).split(',')[1];
    let targets = document.getElementById('target_input').value;

    // Native bridge invocation call directly to python runtime instance
    await google.colab.kernel.invokeFunction('notebook.process_frame_bridge', [base64Image, targets], {});
}

window.renderDetections = function(detections) {
    ctx.clearRect(0, 0, canvas.width, canvas.height);
    let hf = canvas.width / 640;
    let vf = canvas.height / 480;

    if(detections) {
        detections.forEach((o, idx) => {
            let x1 = o.x1, y1 = o.y1, x2 = o.x2, y2 = o.y2;
            let cl = 'hsl(' + ((idx * 137.5) % 360) + ', 85%, 60%)';
            if(o.label.includes('closed eye')) cl = '#f33';
            if(o.label.includes('open eye')) cl = '#3f3';

            ctx.strokeStyle = cl; ctx.lineWidth = 3;
            ctx.strokeRect(x1*hf, y1*vf, (x2-x1)*hf, (y2-y1)*vf);

            ctx.save();
            let textX = x1 * hf;
            let textY = (y1 * vf) - 6;
            ctx.translate(canvas.width - textX, textY);
            ctx.scale(-1, 1);
            ctx.fillStyle = cl; ctx.font = 'bold 14px sans-serif';
            ctx.fillText(o.label.toUpperCase(), 0, 0);
            ctx.restore();
        });
    }
    // Instantly queue next animation frame loop call without delay timers
    if(active) setTimeout(captureCycle, 10);
};
</script>
'''

display(HTML(INTERFACE_HTML))